# CNN/YOLO Training Completion Notebook - Vietnamese Traffic Sign Detection

Notebook này tiếp quản phần còn thiếu của workflow hiện tại: **Model training**, **Model evaluation**, và **Inference / Camera command**.

Bối cảnh đã biết:

- Dữ liệu đã được EDA, cleaning, class remapping và balanced training data preparing.
- Dataset train balanced kỳ vọng: `1701` ảnh / `4368` objects / `52` active classes.
- Lần train trước bị dừng ở epoch 2/50 do `KeyboardInterrupt` nên chưa có `best.pt` hoàn chỉnh.
- Notebook này ưu tiên dùng lại dataset đã chuẩn bị tại `data/data.training.yaml` và lưu trọng số tốt nhất đúng đường dẫn `runs/train/vietnam-traffic-sign-yolo-balanced/weights/best.pt`.


## 1. Cài đặt môi trường và import thư viện

Cell này thiết lập cache nội bộ project, kiểm tra package cần thiết và import các thư viện phục vụ train/evaluate/inference YOLO.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import random
import shutil
import subprocess
import sys
from datetime import datetime

# Xác định thư mục gốc project, kể cả khi notebook được chạy từ thư mục notebooks.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Tạo cache trong project để tránh ghi cấu hình ra ngoài repo.
CACHE_ROOT = PROJECT_ROOT / ".cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("YOLO_CONFIG_DIR", str(CACHE_ROOT / "ultralytics"))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "matplotlib"))
os.environ.setdefault("KAGGLEHUB_CACHE", str(CACHE_ROOT / "kagglehub"))
for folder in ["ultralytics", "matplotlib", "kagglehub"]:
    (CACHE_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Các package tối thiểu cần có để train/evaluate/inference.
REQUIRED_PACKAGES = {
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "yaml": "PyYAML",
    "ultralytics": "ultralytics",
}

missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Đang cài package còn thiếu vào kernel hiện tại:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display
from ultralytics import YOLO

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("YOLO config:", os.environ["YOLO_CONFIG_DIR"])


## 2. Khai báo đường dẫn và helper

Cell này chỉ dùng lại dữ liệu đã chuẩn bị từ workflow trước. Nếu `data/data.training.yaml` chưa tồn tại, notebook sẽ báo lỗi rõ ràng thay vì tự giả định dữ liệu.


In [ ]:
# Các đường dẫn chính của pipeline đã chuẩn bị trước đó.
DATA_ROOT = PROJECT_ROOT / "data"
TRAINING_YAML = DATA_ROOT / "data.training.yaml"
RUNTIME_DATA_YAML = CACHE_ROOT / "data.training.runtime.yaml"
TRAINING_ROOT = DATA_ROOT / "processed" / "vietnamese-traffic-signs-kaggle-training-balanced"

# Cấu hình train đúng theo workflow gốc.
BASE_MODEL = "yolo11n.pt"
FALLBACK_MODEL = "yolo11n.yaml"
EPOCHS = 50
IMGSZ = 640
BATCH = 8
WORKERS = 2
PATIENCE = 30
SEED = 42
RUN_NAME = "vietnam-traffic-sign-yolo-balanced"
PROJECT_DIR = PROJECT_ROOT / "runs" / "train"
RUN_DIR = PROJECT_DIR / RUN_NAME
BEST_WEIGHT = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHT = RUN_DIR / "weights" / "last.pt"
EVAL_PROJECT_DIR = PROJECT_ROOT / "runs" / "eval"
EVAL_RUN_NAME = f"{RUN_NAME}-test"
CONF_THRES = 0.25
SAMPLE_INFERENCE_COUNT = 6
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Helper đọc/ghi YAML theo chuẩn dataset YOLO.
def read_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}

def write_yaml(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")

def normalize_names(names) -> dict[int, str]:
    if isinstance(names, list):
        return {i: str(name) for i, name in enumerate(names)}
    return {int(k): str(v) for k, v in names.items()}

def resolve_split_dir(data_yaml: Path, split: str) -> Path:
    cfg = read_yaml(data_yaml)
    root = Path(cfg.get("path", data_yaml.parent))
    if not root.is_absolute():
        root = (data_yaml.parent / root).resolve()
    split_path = Path(cfg[split])
    return split_path if split_path.is_absolute() else root / split_path

def label_path_for_image(image_path: Path, images_dir: Path) -> Path:
    relative = image_path.relative_to(images_dir)
    return images_dir.parent.parent / "labels" / images_dir.name / relative.with_suffix(".txt")

def read_yolo_label(label_path: Path) -> list[dict]:
    rows = []
    if not label_path.exists():
        return rows
    for raw_line in label_path.read_text(encoding="utf-8").splitlines():
        parts = raw_line.strip().split()
        if len(parts) != 5:
            continue
        try:
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
            rows.append({"class_id": class_id, "x_center": x_center, "y_center": y_center, "width": width, "height": height})
        except ValueError:
            continue
    return rows

def scan_yolo_dataset(data_yaml: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    cfg = read_yaml(data_yaml)
    names_local = normalize_names(cfg["names"])
    image_records, label_records = [], []
    for split in ["train", "val", "test"]:
        images_dir = resolve_split_dir(data_yaml, split)
        image_paths = sorted(path for path in images_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)
        for image_path in image_paths:
            label_path = label_path_for_image(image_path, images_dir)
            labels = read_yolo_label(label_path)
            image_records.append({
                "split": split,
                "image_path": image_path,
                "label_path": label_path,
                "objects": len(labels),
                "has_label_file": label_path.exists(),
            })
            for label in labels:
                class_id = int(label["class_id"])
                label_records.append({
                    "split": split,
                    "image_path": image_path,
                    "label_path": label_path,
                    "class_id": class_id,
                    "class_name": names_local.get(class_id, f"unknown_{class_id}"),
                    **label,
                })
    return pd.DataFrame(image_records), pd.DataFrame(label_records)


## 3. Kiểm tra dữ liệu training đã chuẩn bị

Cell này xác nhận notebook đang dùng đúng dataset balanced đã tạo từ các bước trước và tạo runtime YAML bằng đường dẫn tuyệt đối để Ultralytics đọc ổn định.


In [ ]:
# Không giả định dữ liệu tồn tại: nếu thiếu file YAML hoặc thư mục ảnh/label, dừng sớm với thông báo rõ ràng.
if not TRAINING_YAML.exists():
    raise FileNotFoundError(
        f"Chưa tìm thấy {TRAINING_YAML}. Hãy chạy xong bước Balanced training data preparing trong notebook workflow trước."
    )

training_cfg = read_yaml(TRAINING_YAML)
names = normalize_names(training_cfg["names"])

# Ghi runtime YAML với path tuyệt đối để tránh lỗi khi notebook được chạy từ thư mục khác.
runtime_cfg = dict(training_cfg)
runtime_root = Path(runtime_cfg.get("path", TRAINING_YAML.parent))
if not runtime_root.is_absolute():
    runtime_root = (TRAINING_YAML.parent / runtime_root).resolve()
runtime_cfg["path"] = str(runtime_root).replace("\\", "/")
write_yaml(RUNTIME_DATA_YAML, runtime_cfg)

# Scan dataset để kiểm tra số ảnh/object trước khi train.
images_df, labels_df = scan_yolo_dataset(TRAINING_YAML)
if images_df.empty:
    raise ValueError("Không tìm thấy ảnh nào trong dataset training đã chuẩn bị.")

summary_df = images_df.groupby("split").agg(images=("image_path", "count"), objects=("objects", "sum")).reset_index()
class_count = len(names)
display(summary_df)
print("Số active classes:", class_count)
print("Runtime data YAML:", RUNTIME_DATA_YAML)
print("Best weight sẽ được lưu tại:", BEST_WEIGHT)


## 4. Model Training - huấn luyện YOLO 50 epochs

Cell này huấn luyện YOLO đủ 50 epochs. Nếu đã có `last.pt` từ lần chạy trước nhưng chưa có `best.pt`, code sẽ resume để giảm rủi ro mất tiến độ. Kết quả tốt nhất được lưu tại đúng đường dẫn yêu cầu.


In [ ]:
# Kiểm tra nhanh môi trường torch để biết đang chạy CPU hay GPU.
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Không kiểm tra được torch/CUDA:", exc)

# Nếu có last.pt mà chưa hoàn thành best.pt, resume từ checkpoint để tránh mất tiến độ train cũ.
resume_training = LAST_WEIGHT.exists() and not BEST_WEIGHT.exists()

if resume_training:
    print("Phát hiện last.pt từ lần train trước. Sẽ resume training từ checkpoint:", LAST_WEIGHT)
    model = YOLO(str(LAST_WEIGHT))
else:
    try:
        model = YOLO(BASE_MODEL)
        print("Sử dụng pretrained model:", BASE_MODEL)
    except AttributeError as exc:
        # Fallback cho một số lỗi tương thích Python/Torch khi load checkpoint .pt.
        if "_utils" not in str(exc):
            raise
        print("Không load được pretrained .pt. Chuyển sang kiến trúc YAML:", FALLBACK_MODEL)
        model = YOLO(FALLBACK_MODEL)

# Train model. exist_ok=True giúp ghi đúng run folder quy định thay vì tạo folder mới có hậu tố số.
train_results = model.train(
    data=str(RUNTIME_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    seed=SEED,
    patience=PATIENCE,
    exist_ok=True,
    plots=True,
    resume=resume_training,
)

if not BEST_WEIGHT.exists():
    raise FileNotFoundError(f"Training đã kết thúc nhưng chưa tìm thấy best.pt tại: {BEST_WEIGHT}")

print("Huấn luyện hoàn tất.")
print("Best weight:", BEST_WEIGHT)


## 5. Model Evaluation - đánh giá trên test split

Cell này load `best.pt`, đánh giá trên tập `test`, và lưu Precision, Recall, mAP@0.5, mAP@0.5:0.95 cùng các biểu đồ như confusion matrix/loss curves vào thư mục `runs/eval`.


In [ ]:
if not BEST_WEIGHT.exists():
    raise FileNotFoundError(f"Chưa có best.pt. Hãy chạy cell training trước: {BEST_WEIGHT}")

# Load model tốt nhất sau training.
eval_model = YOLO(str(BEST_WEIGHT))

# Đánh giá trên test split. plots=True yêu cầu Ultralytics xuất confusion matrix và các biểu đồ liên quan.
metrics = eval_model.val(
    data=str(RUNTIME_DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    project=str(EVAL_PROJECT_DIR),
    name=EVAL_RUN_NAME,
    plots=True,
)

evaluation_metrics = {
    "run_name": RUN_NAME,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "best_weight": str(BEST_WEIGHT),
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
}

evaluation_df = pd.DataFrame([evaluation_metrics])
display(evaluation_df)

# Lưu metrics dạng JSON để cell báo cáo cuối notebook có thể đọc lại.
metrics_json_path = EVAL_PROJECT_DIR / EVAL_RUN_NAME / "evaluation_metrics.json"
metrics_json_path.parent.mkdir(parents=True, exist_ok=True)
metrics_json_path.write_text(json.dumps(evaluation_metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print("Đã lưu evaluation metrics:", metrics_json_path)
print("Thư mục biểu đồ evaluation:", metrics_json_path.parent)


## 6. Hiển thị Loss Curves và Confusion Matrix

Ultralytics tự sinh các biểu đồ trong `runs/train/...` và `runs/eval/...`. Cell này tìm các ảnh kết quả thường gặp và hiển thị trực tiếp trong notebook.


In [ ]:
# Danh sách file biểu đồ thường được Ultralytics sinh ra sau train/eval.
plot_candidates = [
    RUN_DIR / "results.png",
    RUN_DIR / "confusion_matrix.png",
    RUN_DIR / "confusion_matrix_normalized.png",
    EVAL_PROJECT_DIR / EVAL_RUN_NAME / "confusion_matrix.png",
    EVAL_PROJECT_DIR / EVAL_RUN_NAME / "confusion_matrix_normalized.png",
    EVAL_PROJECT_DIR / EVAL_RUN_NAME / "P_curve.png",
    EVAL_PROJECT_DIR / EVAL_RUN_NAME / "R_curve.png",
    EVAL_PROJECT_DIR / EVAL_RUN_NAME / "PR_curve.png",
    EVAL_PROJECT_DIR / EVAL_RUN_NAME / "F1_curve.png",
]

found_plots = [path for path in plot_candidates if path.exists()]
if not found_plots:
    print("Chưa tìm thấy biểu đồ. Hãy chạy training/evaluation với plots=True trước.")
else:
    for path in found_plots:
        image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(image)
        plt.axis("off")
        plt.title(path.name)
        plt.show()


## 7. Inference trên ảnh mẫu ngẫu nhiên

Cell này chọn một vài ảnh từ test split, predict bằng `best.pt`, hiển thị bounding box, class name và confidence.


In [ ]:
if not BEST_WEIGHT.exists():
    raise FileNotFoundError(f"Chưa có best.pt. Hãy chạy training trước: {BEST_WEIGHT}")

inference_model = YOLO(str(BEST_WEIGHT))

test_images_dir = resolve_split_dir(TRAINING_YAML, "test")
test_images = sorted(path for path in test_images_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)
if not test_images:
    raise ValueError("Không tìm thấy ảnh test để chạy inference.")

random.seed(SEED)
sample_images = random.sample(test_images, k=min(SAMPLE_INFERENCE_COUNT, len(test_images)))

all_prediction_rows = []
for image_path in sample_images:
    # Predict từng ảnh để dễ hiển thị kết quả và debug khi cần.
    result = inference_model.predict(source=str(image_path), conf=CONF_THRES, imgsz=IMGSZ, verbose=False)[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(f"Inference: {image_path.name}")
    plt.show()

    if result.boxes is not None and len(result.boxes) > 0:
        for class_id, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
            class_id = int(class_id)
            all_prediction_rows.append({
                "image": image_path.name,
                "class_id": class_id,
                "class_name": result.names[class_id],
                "confidence": float(conf),
            })

predictions_df = pd.DataFrame(all_prediction_rows)
if predictions_df.empty:
    print("Không có detection nào trên các ảnh mẫu. Có thể thử giảm CONF_THRES hoặc kiểm tra lại chất lượng model.")
else:
    display(predictions_df.sort_values(["image", "confidence"], ascending=[True, False]))


## 8. Camera Command và hàm demo camera realtime

Cell này chuẩn bị lệnh camera cho demo thực tế, đồng thời cung cấp hàm Python đọc frame từ webcam và predict liên tục.


In [ ]:
# Lệnh CLI đơn giản để chạy camera bằng Ultralytics sau khi đã có best.pt.
camera_command = f'yolo predict model="{BEST_WEIGHT}" source=0 show=True conf={CONF_THRES} imgsz={IMGSZ}'
print("Camera command - chạy trong terminal sau khi training hoàn tất:")
print(camera_command)

def run_camera_demo(model_path: Path = BEST_WEIGHT, camera_index: int = 0, conf: float = CONF_THRES, imgsz: int = IMGSZ):
    """Chạy demo camera realtime bằng OpenCV và YOLO.

    Nhấn phím 'q' để thoát cửa sổ camera.
    """
    if not model_path.exists():
        raise FileNotFoundError(f"Chưa tìm thấy model weight: {model_path}")

    model = YOLO(str(model_path))
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được camera index {camera_index}. Hãy kiểm tra webcam hoặc đổi camera_index.")

    print("Đang chạy camera demo. Nhấn 'q' để thoát.")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Không đọc được frame từ camera.")
                break

            # Predict trực tiếp trên frame BGR từ OpenCV.
            result = model.predict(source=frame, conf=conf, imgsz=imgsz, verbose=False)[0]
            annotated_frame = result.plot()

            cv2.imshow("Vietnam Traffic Sign Detection", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

# Bỏ comment dòng dưới để chạy demo camera trực tiếp trong máy local.
# run_camera_demo()


## 9. Báo cáo tiến độ tự động

Cell cuối này tổng hợp trạng thái thực thi sau khi notebook chạy xong và xuất báo cáo Markdown theo yêu cầu.


In [ ]:
# Gom thông tin thực tế từ file/folder đã sinh ra để báo cáo cuối notebook.
report_lines = [
    "# Báo cáo tiến độ tự động",
    "",
    f"- Thời điểm tạo báo cáo: `{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}`",
    f"- Dataset runtime: `{RUNTIME_DATA_YAML}`",
]

# Báo cáo dữ liệu đã chuẩn bị.
try:
    latest_images_df, latest_labels_df = scan_yolo_dataset(TRAINING_YAML)
    latest_summary = latest_images_df.groupby("split").agg(images=("image_path", "count"), objects=("objects", "sum")).reset_index()
    for _, row in latest_summary.iterrows():
        report_lines.append(f"- Split `{row['split']}`: `{int(row['images'])}` ảnh / `{int(row['objects'])}` objects")
    report_lines.append(f"- Số active classes: `{len(names)}`")
except Exception as exc:
    report_lines.append(f"- Chưa đọc được thống kê dataset: `{exc}`")

# Báo cáo training và trọng số.
if BEST_WEIGHT.exists():
    report_lines.append(f"- Đã xuất file trọng số tốt nhất thành công: `{BEST_WEIGHT}`")
else:
    report_lines.append(f"- Chưa tìm thấy file trọng số tốt nhất tại: `{BEST_WEIGHT}`")

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    try:
        results_df = pd.read_csv(results_csv)
        completed_epochs = len(results_df)
        report_lines.append(f"- Đã hoàn thành huấn luyện `{completed_epochs}` epochs theo `results.csv`.")
        if completed_epochs >= EPOCHS:
            report_lines.append(f"- Đã hoàn thành mục tiêu huấn luyện `{EPOCHS}` epochs.")
        else:
            report_lines.append(f"- Chưa đủ mục tiêu `{EPOCHS}` epochs; cần chạy tiếp/resume training.")
    except Exception as exc:
        report_lines.append(f"- Có `results.csv` nhưng chưa đọc được số epoch: `{exc}`")
else:
    report_lines.append("- Chưa tìm thấy `results.csv`, chưa xác nhận được số epoch đã train.")

# Báo cáo evaluation metrics nếu đã chạy cell evaluation.
metrics_json_path = EVAL_PROJECT_DIR / EVAL_RUN_NAME / "evaluation_metrics.json"
if metrics_json_path.exists():
    eval_metrics = json.loads(metrics_json_path.read_text(encoding="utf-8"))
    report_lines.extend([
        f"- Precision: `{eval_metrics.get('precision', float('nan')):.4f}`",
        f"- Recall: `{eval_metrics.get('recall', float('nan')):.4f}`",
        f"- mAP@0.5: `{eval_metrics.get('mAP50', float('nan')):.4f}`",
        f"- mAP@0.5:0.95: `{eval_metrics.get('mAP50_95', float('nan')):.4f}`",
    ])
else:
    report_lines.append("- Chưa có evaluation metrics JSON; hãy chạy cell Model Evaluation sau khi có `best.pt`.")

# Báo cáo inference/camera readiness.
if BEST_WEIGHT.exists():
    report_lines.append(f"- Inference sẵn sàng với confidence threshold `{CONF_THRES}`.")
    report_lines.append(f"- Camera command: `{camera_command}`")
else:
    report_lines.append("- Inference/camera chưa sẵn sàng vì thiếu `best.pt`.")

display(Markdown("\n".join(report_lines)))
